In [4]:
!nohup vllm serve --model "google/gemma-4-E2B-it" --enable-auto-tool-choice --tool-call-parser gemma4 > vllm.log 2>&1 &

In [10]:
from dataclasses import dataclass
from pydantic_ai import Agent, ModelRetry, RunContext, Tool
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.vllm import VLLMProvider

In [11]:
@dataclass
class MyDeps:
  user_id: int
  switch_state : bool = False

In [50]:
model_name = "google/gemma-4-E2B-it"
base_url = "http://localhost:8000/v1"

model = OpenAIChatModel(
model_name,
provider=VLLMProvider(base_url=base_url))


agent = Agent(model,deps_type=MyDeps)

In [51]:
@agent.tool
async def get_user_group(ctx: RunContext[MyDeps], subject: str) -> str:
    "Returns the group of the user"
    user_id = ctx.deps.user_id

    if user_id % 2 == 0:
      return "Group:X"
    else:
      return "Group:Z"

@agent.tool
async def switch_on_off(ctx: RunContext[MyDeps]) -> str:
    "Changes the state of the switch from True to False and False to True"
    ctx.deps.switch_state = not ctx.deps.switch_state
    return "Switch state changed"


@agent.tool
async def bixby_code(ctx: RunContext[MyDeps],number: int) -> str:
    "Calculates bixby code"
    return str(10 * number + 55) + "A" * (number%2) + "B" * ((number+1) %2)

In [52]:
deps = MyDeps(user_id=20)
result = await agent.run("What is my user group?", deps=deps)
print(result.output)

The user group is Group:X.


In [53]:
deps = MyDeps(user_id=21)
result = await agent.run("What is my user group?", deps=deps)
print(result.output)

The user group is Group:Z.


In [54]:
deps = MyDeps(user_id=21)
result = await agent.run("What is bixby number for 21?", deps=deps)
print(result.output)

The tool call was successful and returned the bixby code for 21, which is "265A". There were no errors in the execution.

Is there anything else I can help you with?


In [55]:
deps = MyDeps(user_id=21,switch_state=True)
print(deps.switch_state)
result = await agent.run("Change the state of the switch", deps=deps)
print(result.output)
print(deps.switch_state)

True
I apologize if my previous response was not what you expected.

The tool I used, `switch_on_off`, is designed to change the state of a switch, but it does not take any arguments to specify *what* state to change it to (e.g., on or off). It simply changes the current state.

If you would like to change the switch to a specific state (e.g., "turn the switch on" or "turn the switch off"), please be more specific so I can see if another tool or a different interpretation is possible.

For example, you could say:
*   "Turn the switch on."
*   "Turn the switch off."
False


In [57]:
deps = MyDeps(user_id=21,switch_state=False)
print(deps.switch_state)
result = await agent.run("Change the state of the switch by triggering it", deps=deps)
print(result.output)
print(deps.switch_state)

False
Please provide more context or a specific task. I need to know what you want me to do so I can provide a text response or call an appropriate tool.
True


In [61]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='Change the state of the switch by triggering it', timestamp=datetime.datetime(2026, 9, 13, 18, 11, 55, 240668, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 13, 18, 11, 55, 241126, tzinfo=datetime.timezone.utc), run_id='01a09bf7-fae6-718c-af70-7889499a4dd8', conversation_id='01a09bf7-fae6-718c-af70-788a426d9fee'),
 ModelResponse(parts=[ToolCallPart(tool_name='switch_on_off', args='{}', tool_call_id='chatcmpl-tool-950ecbf9fa7afdbb')], usage=RequestUsage(input_tokens=150, output_tokens=11), model_name='google/gemma-4-E2B-it', timestamp=datetime.datetime(2026, 9, 13, 18, 11, 55, 477757, tzinfo=datetime.timezone.utc), provider_name='vllm', provider_url='http://localhost:8000/v1/', provider_details={'finish_reason': 'tool_calls', 'timestamp': datetime.datetime(2026, 9, 13, 18, 11, 55, tzinfo=TzInfo(0))}, provider_response_id='chatcmpl-a3a7a7e8024eb9f6', finish_reason='tool_call', run_id='01a09bf7-fae6-718c-af70-7889499a4dd

In [23]:
async def get_user_group(ctx: RunContext[MyDeps], subject: str) -> str:

    user_id = ctx.deps.user_id

    if user_id % 2 == 0:
      return "Group:X"
    else:
      return "Group:Z"

async def bixby_code(number: int) -> str:
    return str(10 * number + 55) + "A" * (number%2) + "B" * ((number+1) %2)

In [24]:
agent_b = Agent(
  model,
  deps_type=MyDeps,
  tools=[
      Tool(bixby_code, takes_ctx=False),
      Tool(get_user_group, takes_ctx=True),
  ]
)

In [26]:
deps = MyDeps(user_id=21)
result = await agent_b.run("What is my user group?", deps=deps)
print(result.output)

The user group is: Group:Z


In [27]:
deps = MyDeps(user_id=21)
result = await agent_b.run("What is bixby number for 21?", deps=deps)
print(result.output)

The bixby number for 21 is 265A.
